# Introduction

Although the "O" in PyGOM stands for **O**rdinary-Differential-Equations, the package is fundamentally built around compartmental models.

A compartmental model describes how entities move between different states or categories.
The entities may be people in an epidemic model, molecules in a chemical reaction system, or money in an economic context.
A defining (and simplifying) feature of compartmental models is that rather than tracking every individual entity, we track the total quantity present in each state.

The system may change via the occurance of transitions


The state of the system is described by the collection of these populations and is represented by the *state vector*: $\boldsymbol{y}$

```{note}
# SIR Example

For example, in the SIR model:

$
\begin{aligned}
\mathbf{y} = (S, I, R)
\end{aligned}
$

where S, I and R give the total number of Susceptible, Infected and Recovered indivduals respectively.
```

To describe how the system evolves, we define a set of events.
Each event occurs at some rate and induces a change in the state vector.
When event, j, occurs, the system changes as:

$$\begin{aligned}
\boldsymbol{y} \rightarrow \boldsymbol{y} + \boldsymbol{v}_j
\end{aligned}$$

The vectors $\boldsymbol{v}_j$​ are called *state-change vectors* and together form the *state-change matrix* (or *stoichiometry matrix*)

$$\begin{aligned}
\mathcal{D} = [\boldsymbol{v}_1, \ldots \boldsymbol{v}_{N_E}]
\end{aligned}$$
where there are $N_E$ total events.

```{note}
# SIR Example

In the SIR model there are 2 events: Infection and Recovery.
An infection event moves an individual from the S to the I compartment and a recovery moves from I to R.

$
\begin{aligned}
\mathcal{D} =
\begin{pmatrix}
-1 & 0\\
1 & -1\\
0 & 1\\
\end{pmatrix}
\end{aligned}
$
```

Each event occurs at an associated rate $\lambda_j$​, and these rates are collected into the event-rate vector

$$\begin{aligned}
\boldsymbol{\lambda} = [\lambda_1, \ldots \lambda_{N_E}]
\end{aligned}$$

```{note}
# SIR Example

In the SIR model, the infection and recovery events occur at the following rates:

$
\begin{aligned}
\boldsymbol{\lambda} = \left( \frac{\beta S I}{N}, \gamma I \right)
\end{aligned}
$
```

The state vector $\boldsymbol{y}$, state-change matrix $\mathcal{D}$, and event-rate vector $\boldsymbol{\lambda}$ together define a compartmental model in PyGOM.

# Representation

This defining set of information is often represented graphiclaly.
In this way, nodes indicate the states and arrows (directed edges) indicate the transfer of individuals between them.
The arrows are then annotated with the rate at which the transitions occur and it is implied that a transition event leads to the transfer of one individual.
This typically works when we have one transition per event.

In [ ]:
from graphviz import Digraph

dot = Digraph()

dot.body.extend(['rankdir=LR'])

states=['S', 'I', 'R']

for s in states:
    dot.node(s)

dot.attr(size='5,5')

dot.edge('S', 'I', label='&beta;SI/N')
dot.edge('I', 'R', label='&gamma;I')

dot

<!-- ```{note}
For those unfamiliar with the SIR model, an outline can be found in the {doc}`common models <common_models/SIR>` section.
``` -->

# Transitions

It is worth distinguisihgn Transitions from Events.
An event can have multiple consittuent.
An event can trigger multiple transitions when it occurs.
We now talk about different types of transitions:

## Between state

So far we have only considered transitions where individuals move from one state to another

## In-to and out-of the system

This could appear in the SIR model through births and deaths, for example.
Here, we allow individuals to be born in a susceptible state at a rate, $\mu$, proportional to the total population, $N$.
We also have deaths occurring from each compartment at the same rate, proportional to the population of the respective compartment.

To indicate these graphically, birth and death processes lack an origin or a destination state respectively:

In [ ]:
dot = Digraph()

dot.body.extend(['rankdir=LR'])

states=['S', 'I', 'R']

for s in states:
    dot.node(s)

dot.edge('S', 'I', label='&beta;SI/N')
dot.edge('I', 'R', label='&gamma;I')

dot.node("bornS", label="", shape='none', height="0", width="0")
dot.edge('bornS', 'S', label="&mu;N")

dot.node("deadS", label="", shape="none", height="0", width="0")
dot.edge('S', 'deadS', label="&mu;S")

dot.node("deadI", label="", shape="none", height="0", width="0")
dot.edge('I', 'deadI', label="&mu;I")

dot.node("deadR", label="", shape="none", height="0", width="0")
dot.edge('R', 'deadR', label="&mu;R")

dot

```{note}
We may actually be interested in tracking the total number of deaths, in which case we would define a new compartment, say $D$, and have this as the destination state for the death processes. 
```

## Multi-transition events

It is possible for an event to trigger multiple transitions.
For instance, let's examine what happens if we wish to track the cumulative number of infections.
Every time an infection occurs, as well as an individual transferring from $S$ to $I$, the $I_{tot}$ compartment increments by 1 too.
We show this graphically as a birth process into $I_{tot}$ and indicate that it is correlated with the $S\rightarrow I$ transition by a shared colour (the default colour, black, does not imply correlation so that recovery, births and deaths are independent processes).
Correlated transitions have the same underlying rate, so that, strictly speaking, only one rate needs be specified along the blue edges.
However, it can help for clarity to include the $\frac{\beta S I}{N}$ in both cases, as we have done here.

In [ ]:
dot = Digraph()

dot.body.extend(['rankdir=LR'])

states=['S', 'I', 'R']

for s in states:
    dot.node(s)

dot.edge('S', 'I', label='&beta;SI/N', color="blue")
dot.edge('I', 'R', label='&gamma;I')

dot.node("bornS", label="", shape="none", height="0", width="0")
dot.edge('bornS', 'S', label="&mu;N")

dot.node("deadS", label="", shape="none", height="0", width="0")
dot.edge('S', 'deadS', label="&mu;S")

dot.node("deadI", label="", shape="none", height="0", width="0")
dot.edge('I', 'deadI', label="&mu;I")

dot.node("deadR", label="", shape="none", height="0", width="0")
dot.edge('R', 'deadR', label="&mu;R")

dot.node("It", label="", shape="none", height="0", width="0")
dot.edge('It', 'Iₜₒₜ', label='&beta;SI/N', color="blue")

dot

## Non-unit transitions

The final feature we should account for is the magnitude of transitions.
So far, we have only dealt with transitions which result in the state populations changing by $\pm 1$.
This is typically the case for epidemic compartmental models, where we are dealing with the movement of individuals.
We could imagine scenarios, such as vaccinations being administered in batches of, say, 100, meaning that in one vaccination round 100 individuals will move from $S$ to $R$ in one go.
Or, perhaps, if we restrict our study a cohort of twins then each birth event increments the $S$ population by 2.

It is more common, however, to find these kinds of situations in the study chemical systems.
For example, if we were tracking populations of $\mathrm{H}_2$, $\mathrm{O}_2$ and $\mathrm{H}_2\mathrm{O}$ in the synthesis of water, then one reaction results in population changes: $\mathrm{H}_2$: -2, $\mathrm{O}_2$: -1 and $\mathrm{H}_2\mathrm{O}$ +2.

In our SIR model, let's say we wish to track the monetary cost of the epidemic, $M$.
If each infection costs $c$ monetary units then we must add another transition to our model, which is triggered in concert with infection....
This time, we must indicate that the transition has a different magnitude.
Graphically we do so by adding a label of the amount to the arrowhead.

In [ ]:
from graphviz import Digraph


dot = Digraph()

dot.body.extend(['rankdir=LR'])

states=['S', 'I', 'R']

for s in states:
    dot.node(s)

dot.edge('S', 'I', label='&beta;SI/N', color="blue")
dot.edge('I', 'R', label='&gamma;I')

dot.node("bornS", label="", shape="none", height="0", width="0")
dot.edge('bornS', 'S', label="&mu;N")

dot.node("deadS", label="", shape="none", height="0", width="0")
dot.edge('S', 'deadS', label="&mu;S")

dot.node("deadI", label="", shape="none", height="0", width="0")
dot.edge('I', 'deadI', label="&mu;I")

dot.node("deadR", label="", shape="none", height="0", width="0")
dot.edge('R', 'deadR', label="&mu;R")

dot.node("It", label="", shape="none", height="0", width="0")
dot.edge('It', 'Iₜₒₜ', label='&beta;SI/N', color="blue")

dot.node("cost", label="", shape="none", height="0", width="0")
dot.edge('cost', 'M', label='&beta;SI/N', color="blue", headlabel='c')

dot

We will now see how to build these models in PyGOM